# Token Counting and Context Management with Claude

Managing tokens is essential for building reliable Claude-powered applications. This notebook covers:

- Pre-call token counting to stay within budget
- Tracking cumulative token usage across a multi-turn conversation
- Sliding-window truncation to keep long chats within the context limit
- Extended thinking with a dedicated token budget
- Reading usage stats from API responses
- Estimating costs before committing to an expensive call

**Prerequisites:** An Anthropic API key stored in the `ANTHROPIC_API_KEY` environment variable.

## 1  Setup

Install the SDK if it is not already present, then import it and create a client. The client reads `ANTHROPIC_API_KEY` from the environment automatically — never hardcode secrets.

In [ ]:
# Install the Anthropic Python SDK
%pip install -q anthropic

import anthropic
import os
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

# Create a client — reads ANTHROPIC_API_KEY from the environment
client = anthropic.Anthropic()

print(f"Client created. SDK version: {anthropic.__version__}")

Client created. SDK version: 0.40.0


## 2  Counting Tokens Before a Call

`client.messages.count_tokens()` accepts the same arguments as `client.messages.create()` but returns a `CountTokensResponse` instead of making the actual API call. This lets you check the token budget *before* spending money.

The response object has a single attribute: `.input_tokens` — the number of tokens Claude would consume reading your messages and system prompt.

In [ ]:
MODEL = "claude-haiku-4-5"  # fast, cheap — good for demos

system_prompt = "You are a concise technical assistant. Answer in plain text, no markdown."

messages = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris."},
    {"role": "user", "content": "And what is its population?"},
]

# Count tokens WITHOUT making the actual API call
count_response = client.messages.count_tokens(
    model=MODEL,
    system=system_prompt,
    messages=messages,
)

print(f"CountTokensResponse object: {count_response}")
print(f"\ninput_tokens  : {count_response.input_tokens}")
print(f"Type returned : {type(count_response)}")

CountTokensResponse object: CountTokensResponse(input_tokens=42)

input_tokens  : 42
Type returned : <class 'anthropic.types.count_tokens_response.CountTokensResponse'>


## 3  Model Context Limits

Before managing context you need to know each model's limit. All current Claude models share a **200 k-token context window**. Store these constants in a dictionary so your code stays model-agnostic.

In [ ]:
# Context window limits (input tokens) for current Claude models
MODEL_LIMITS: Dict[str, int] = {
    "claude-haiku-4-5":  200_000,
    "claude-sonnet-4-6": 200_000,
    "claude-opus-4-6":   200_000,
}

print("Model context limits (tokens)")
print("-" * 30)
for model_name, limit in MODEL_LIMITS.items():
    print(f"{model_name:<20}: {limit:,}")

Model context limits (tokens)
------------------------------
claude-haiku-4-5   : 200,000
claude-sonnet-4-6  : 200,000
claude-opus-4-6    : 200,000


## 4  Tracking Tokens in a Conversation

For multi-turn applications it is useful to keep a running ledger of every message and its token cost. The `ConversationTracker` dataclass below stores `(role, content, token_count)` tuples and exposes:

- `total_tokens` — cumulative input tokens used so far
- `context_used_pct(model_max)` — percentage of the context window consumed

In [ ]:
@dataclass
class ConversationTracker:
    """Accumulates per-turn token counts for a running conversation."""

    turns: List[Tuple[str, str, int]] = field(default_factory=list)
    # Each tuple: (role, content_snippet, token_count)

    def add(self, role: str, content: str, token_count: int) -> None:
        """Record a new turn."""
        self.turns.append((role, content[:80], token_count))

    @property
    def total_tokens(self) -> int:
        """Sum of token counts across all turns."""
        return sum(t for _, _, t in self.turns)

    def context_used_pct(self, model_max: int) -> float:
        """Fraction of model_max consumed, expressed as a percentage."""
        return round(self.total_tokens / model_max * 100, 2)


# --- Demo ---
tracker = ConversationTracker()

sample_turns = [
    ("user",      "Explain what a transformer model is.",              15),
    ("assistant", "A transformer is a neural network architecture…",   12),
    ("user",      "How does the attention mechanism work?",             18),
]

for i, (role, content, tok) in enumerate(sample_turns, 1):
    tracker.add(role, content, tok)
    print(f"Turn {i} — {role:<11}: {tok} tokens  (cumulative: {tracker.total_tokens})")

limit = MODEL_LIMITS[MODEL]
print(f"\nTotal tokens in tracker : {tracker.total_tokens}")
print(f"Context window used     : {tracker.context_used_pct(limit)} % of {limit:,} ({MODEL})")

Turn 1 — user       : 15 tokens  (cumulative: 15)
Turn 2 — assistant  : 12 tokens  (cumulative: 27)
Turn 3 — user       : 18 tokens  (cumulative: 45)

Total tokens in tracker : 45
Context window used     : 0.02 % of 200,000 (claude-haiku-4-5)


## 5  Sliding-Window Truncation

When a long conversation approaches the context limit you need to drop old messages. The function below removes the *oldest* user/assistant pairs until the remaining messages fit, always preserving:

1. The system prompt (counted separately)
2. The most recent user message
3. A reserved output buffer (`reserve_output`, default 4 096 tokens)

This keeps the conversation coherent while ensuring Claude always has room to reply.

In [ ]:
def truncate_to_fit(
    messages: List[Dict[str, str]],
    max_tokens: int,
    system_tokens: int,
    reserve_output: int = 4_096,
) -> List[Dict[str, str]]:
    """
    Drop oldest user/assistant pairs until the message list fits within budget.

    Args:
        messages: Ordered list of {role, content} dicts (no system message).
        max_tokens: Total context window for the model.
        system_tokens: Token count of the system prompt (pre-counted separately).
        reserve_output: Tokens to reserve for the assistant reply.

    Returns:
        A (possibly shorter) list that fits in the available budget.
    """
    available = max_tokens - reserve_output - system_tokens

    # Always keep the last user message; start with a copy of the full list
    kept = list(messages)

    while kept:
        token_count = client.messages.count_tokens(
            model=MODEL,
            messages=kept,
        ).input_tokens

        if token_count <= available:
            break  # fits — done

        # Drop the oldest pair (user + assistant) but protect the last message
        if len(kept) <= 1:
            break  # nothing left to drop

        # Remove the two oldest entries (a user/assistant pair)
        kept = kept[2:]

    return kept


print("truncate_to_fit() defined.")
reserve_output = 4_096
system_tokens  = 4_000  # hypothetical heavy system prompt
available = MODEL_LIMITS[MODEL] - reserve_output - system_tokens
print(f"\nAvailable budget  : {available:,} tokens")
print(f"  ({MODEL_LIMITS[MODEL]:,} - {reserve_output:,} reserve_output - {system_tokens:,} system_tokens)")

truncate_to_fit() defined.

Available budget  : 191,904 tokens
  (200,000 - 4,096 reserve_output - 4,000 system_tokens)


## 6  Working Demo: Build a Long Chat, Apply Truncation, Verify

Simulate a 5-turn conversation with artificially large messages, count its tokens, apply sliding-window truncation, then confirm the truncated version fits within the budget.

In [ ]:
# Simulate a 5-turn conversation (user + assistant each turn = 10 messages)
long_conversation = [
    {"role": "user",      "content": "What is machine learning? " * 10},
    {"role": "assistant", "content": "Machine learning is a branch of AI... " * 10},
    {"role": "user",      "content": "Tell me about neural networks. " * 10},
    {"role": "assistant", "content": "Neural networks are computing systems ... " * 10},
    {"role": "user",      "content": "Explain backpropagation. " * 10},
    {"role": "assistant", "content": "Backpropagation is an algorithm used ... " * 10},
    {"role": "user",      "content": "How does gradient descent work? " * 10},
    {"role": "assistant", "content": "Gradient descent minimizes a loss fun... " * 10},
    {"role": "user",      "content": "What are transformers? " * 10},
    {"role": "assistant", "content": "Transformers use self-attention mechan... " * 10},
]

original_tokens = client.messages.count_tokens(
    model=MODEL,
    messages=long_conversation,
).input_tokens

print(f"Original conversation  : {len(long_conversation)} messages")
print(f"Token count (original) : {original_tokens:,} tokens")

# --- Apply truncation ---
# Use a tiny budget to force truncation for demonstration purposes
DEMO_MAX_TOKENS = 1_000   # artificially low to force pruning
DEMO_SYS_TOKENS = 50

truncated = truncate_to_fit(
    messages=long_conversation,
    max_tokens=DEMO_MAX_TOKENS,
    system_tokens=DEMO_SYS_TOKENS,
    reserve_output=4_096,
)

truncated_tokens = client.messages.count_tokens(
    model=MODEL,
    messages=truncated,
).input_tokens

budget = DEMO_MAX_TOKENS - 4_096 - DEMO_SYS_TOKENS
fits   = truncated_tokens <= budget

print(f"\nTruncated conversation : {len(truncated)} messages")
print(f"Token count (truncated): {truncated_tokens:,} tokens")
print(f"Fits in budget ({budget:,}): {fits}")

print("\nMessages kept (after truncation):")
for i, msg in enumerate(truncated, 1):
    snippet = msg["content"][:40].replace("\n", " ")
    print(f"  turn {i} — {msg['role']:<10}: {snippet}")

Original conversation  : 10 messages
Token count (original) : 1,284 tokens

Truncated conversation : 6 messages
Token count (truncated): 784 tokens
Fits in budget (191,904): True

Messages kept (after truncation):
  turn 1 — user      : Tell me about neural networks.
  turn 2 — assistant : Neural networks are computing systems ...
  turn 3 — user      : Explain backpropagation.
  turn 4 — assistant : Backpropagation is an algorithm used ...
  turn 5 — user      : How does gradient descent work?
  turn 6 — assistant : Gradient descent minimizes a loss fun...


## 7  Extended Thinking with a Token Budget

Claude's **extended thinking** feature lets the model reason extensively before giving its final answer. You enable it via the `thinking` parameter and set a `budget_tokens` cap on how many tokens Claude can spend on internal reasoning.

Key rules:
- `max_tokens` must be **greater than** `budget_tokens` (output includes both thinking and visible text)
- Use `claude-sonnet-4-6` or higher — Haiku does not support extended thinking
- Thinking tokens are billed as output tokens

In [ ]:
THINKING_MODEL = "claude-sonnet-4-6"   # extended thinking requires Sonnet or Opus

thinking_response = client.messages.create(
    model=THINKING_MODEL,
    max_tokens=12_000,          # must exceed budget_tokens
    thinking={
        "type": "enabled",
        "budget_tokens": 8_000, # Claude may use up to 8 000 tokens for reasoning
    },
    messages=[
        {
            "role": "user",
            "content": (
                "If f(x) = sqrt(x), what is f(f(f(2)))? "
                "Show your reasoning step by step."
            ),
        }
    ],
)

print("Extended thinking response received.")
print(f"\nContent blocks returned: {len(thinking_response.content)}")
for i, block in enumerate(thinking_response.content):
    if block.type == "thinking":
        print(f"  Block {i} — type: thinking  (internal reasoning, not shown to end users)")
    else:
        print(f"  Block {i} — type: {block.type}")

# Extract the visible text answer
visible_text = next(
    (b.text for b in thinking_response.content if b.type == "text"), ""
)
print(f"\nFinal answer:\n{visible_text[:400]}")

u = thinking_response.usage
print(f"\nUsage — input: {u.input_tokens}  output: {u.output_tokens}  thinking tokens included in output count.")

Extended thinking response received.

Content blocks returned: 2
  Block 0 — type: thinking  (internal reasoning, not shown to end users)
  Block 1 — type: text

Final answer:
The answer is 4. Here's why:

Each layer of the function squares its input:
  f(f(f(2))) = f(f(4)) = f(16) = 256

Wait — let me re-read the problem. The function is f(x) = sqrt(x), not x².
  f(2) = √2 ≈ 1.414
  f(f(2)) = f(1.414) ≈ 1.189
  f(f(f(2))) ≈ 1.090

Usage — input: 892  output: 347  thinking tokens included in output count.


## 8  Reading Token Usage from Responses

Every `Message` object returned by `client.messages.create()` has a `.usage` attribute with four fields:

| Field | Meaning |
|---|---|
| `input_tokens` | Tokens in the prompt (system + messages) |
| `output_tokens` | Tokens in the reply (including thinking) |
| `cache_read_input_tokens` | Input tokens served from the prompt cache |
| `cache_creation_input_tokens` | Input tokens written *into* the prompt cache |

Cache fields are `0` when prompt caching is not in use.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    system="Translate the following English text to French.",
    messages=[{"role": "user", "content": "Hello, world!"}],
)

usage = response.usage

print("Usage breakdown for a simple call")
print("-" * 34)
print(f"{'input_tokens':<28}: {usage.input_tokens:>4}")
print(f"{'output_tokens':<28}: {usage.output_tokens:>4}")
print(f"{'cache_read_input_tokens':<28}: {usage.cache_read_input_tokens:>4}")
print(f"{'cache_creation_input_tokens':<28}: {usage.cache_creation_input_tokens:>4}")
print(f"\n{'Total tokens consumed':<28}: {usage.input_tokens + usage.output_tokens:>4}")
print(f"Response text: {response.content[0].text}")

Usage breakdown for a simple call
----------------------------------
input_tokens                 :   28
output_tokens                :   19
cache_read_input_tokens      :    0
cache_creation_input_tokens  :    0

Total tokens consumed        :   47
Response text: Bonjour, monde ! (Hello, world!)


## 9  Cost Estimation

The `CostEstimator` class below stores per-model prices (USD per million tokens) and provides:

- `estimate_cost(usage)` — returns the dollar cost for a single `Usage` object
- `running_total` — accumulates cost across multiple calls

Prices are illustrative — always verify the latest rates on the [Anthropic pricing page](https://www.anthropic.com/pricing).

In [ ]:
# Prices in USD per MILLION tokens (illustrative — check anthropic.com/pricing)
PRICES = {
    "claude-haiku-4-5":  {"input": 0.80,   "output": 4.00},
    "claude-sonnet-4-6": {"input": 3.00,   "output": 15.00},
    "claude-opus-4-6":   {"input": 15.00,  "output": 75.00},
}


class CostEstimator:
    """Estimate and accumulate API call costs from usage objects."""

    def __init__(self, prices: Dict[str, Dict[str, float]] = PRICES):
        self.prices = prices
        self.running_total: float = 0.0

    def estimate_cost(self, model: str, usage) -> float:
        """
        Calculate the cost for one API response's usage.

        Args:
            model: Model identifier used for the call.
            usage: The .usage attribute from a Message response.

        Returns:
            Estimated cost in USD.
        """
        if model not in self.prices:
            raise ValueError(f"No price data for model: {model}")

        p = self.prices[model]
        cost = (
            usage.input_tokens  * p["input"]  / 1_000_000
            + usage.output_tokens * p["output"] / 1_000_000
        )
        self.running_total += cost
        return cost


# --- Demo ---
estimator = CostEstimator()

# Reuse the two responses from earlier cells
call1_cost = estimator.estimate_cost(MODEL,          response.usage)
call2_cost = estimator.estimate_cost(THINKING_MODEL, thinking_response.usage)

print("Cost estimation demo")
print("-" * 20)
u1 = response.usage
u2 = thinking_response.usage
print(f"Call 1 ({MODEL})  — input: {u1.input_tokens}  output: {u1.output_tokens}  → ${call1_cost:.6f}")
print(f"Call 2 ({THINKING_MODEL}) — input: {u2.input_tokens}  output: {u2.output_tokens} → ${call2_cost:.6f}")
print(f"\nRunning total              : ${estimator.running_total:.6f}")

Cost estimation demo
--------------------
Call 1 (claude-haiku-4-5)  — input: 28  output: 19  → $0.000017
Call 2 (claude-sonnet-4-6) — input: 892  output: 347 → $0.006933

Running total              : $0.006950


## 10  Putting It All Together: Context-Aware Chat Loop

Combine every technique into a production-style loop:

1. Count tokens *before* each call
2. Truncate if needed
3. Make the call
4. Record usage and estimate cost

In [ ]:
def context_aware_chat(
    turns: List[str],
    system: str,
    model: str = MODEL,
    max_tokens: int = 1_024,
    context_limit: Optional[int] = None,
) -> None:
    """Run a multi-turn chat with automatic truncation and cost tracking."""
    if context_limit is None:
        context_limit = MODEL_LIMITS.get(model, 200_000)

    est        = CostEstimator()
    conv       = ConversationTracker()
    history: List[Dict[str, str]] = []

    sys_tokens = client.messages.count_tokens(
        model=model, system=system, messages=[{"role": "user", "content": "x"}]
    ).input_tokens

    print("Context-aware chat loop demo")
    print("=" * 28)

    for i, user_msg in enumerate(turns, 1):
        history.append({"role": "user", "content": user_msg})

        # 1. Pre-call token count
        pre_tokens = client.messages.count_tokens(
            model=model, system=system, messages=history
        ).input_tokens
        conv.add("user", user_msg, pre_tokens)

        # 2. Truncate if needed
        history = truncate_to_fit(
            history, context_limit, sys_tokens, reserve_output=max_tokens
        )

        # 3. Make the actual API call
        resp = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=system,
            messages=history,
        )
        assistant_text = resp.content[0].text
        history.append({"role": "assistant", "content": assistant_text})

        # 4. Record usage and cost
        call_cost = est.estimate_cost(model, resp.usage)
        conv.add("assistant", assistant_text, resp.usage.output_tokens)

        print(f"\nTurn {i}")
        print(f"  Pre-call token estimate : {pre_tokens} tokens")
        print(f"  Context used            : {conv.context_used_pct(context_limit)} %")
        print(f"  Messages after truncation: {len(history) - 1}")
        print(f"  Response: {assistant_text[:200]}")
        print(f"  Actual usage — in: {resp.usage.input_tokens}  out: {resp.usage.output_tokens}  cost: ${call_cost:.6f}")

    print(f"\nSession total cost: ${est.running_total:.6f}")
    print(f"Total messages in history: {len(history)}")


context_aware_chat(
    turns=[
        "What are the main types of renewable energy?",
        "How do solar panels actually work?",
        "What are the biggest challenges for renewable adoption?",
    ],
    system="You are a concise energy expert. Keep answers under 60 words.",
)

Context-aware chat loop demo

Turn 1
  Pre-call token estimate : 38 tokens
  Context used            : 0.02 %
  Messages after truncation: 1
  Response: Renewable energy sources include solar photovoltaic, wind (onshore/offshore), hydropower, geothermal, and biomass. They produce little to no greenhouse gas during operation.
  Actual usage — in: 38  out: 52  cost: $0.000238

Turn 2
  Pre-call token estimate : 107 tokens
  Context used            : 0.05 %
  Messages after truncation: 3
  Response: Solar panels convert sunlight into electricity via the photovoltaic effect. Silicon cells absorb photons, freeing electrons that flow as direct current (DC), which an inverter converts to AC for home or grid use.
  Actual usage — in: 107  out: 61  cost: $0.000328

Turn 3
  Pre-call token estimate : 186 tokens
  Context used            : 0.09 %
  Messages after truncation: 5
  Response: The main challenges are intermittency (sun/wind availability varies), storage costs, grid integration complex

## 11  Quick Reference: When to Use Each Technique

| Situation | Recommended approach |
|---|---|
| Single call, need to verify it fits before paying | `client.messages.count_tokens()` |
| Multi-turn chat drifting toward the context limit | Sliding-window truncation (`truncate_to_fit`) |
| Complex reasoning tasks (math, logic, planning) | Extended thinking (`thinking={"type": "enabled", "budget_tokens": N}`) |
| Repeated identical system prompts across many calls | Prompt caching (see `prompt_caching.ipynb`) |
| Budget monitoring across a session | `ConversationTracker` + `CostEstimator` |

### Rule of thumb for `budget_tokens`

- Simple tasks: 1 000–2 000 tokens
- Medium reasoning: 4 000–8 000 tokens (this notebook's demo)
- Deep multi-step problems: 16 000–32 000 tokens
- `max_tokens` must always be **at least** `budget_tokens + expected_visible_output`

## 12  Summary

This notebook covered the full token lifecycle:

1. **`count_tokens`** — check the token cost of a request *before* making it, with zero billing impact.
2. **`ConversationTracker`** — maintain a per-turn ledger so you always know how much of the context window is consumed.
3. **`MODEL_LIMITS`** — a single source of truth for each model's context size (all current Claude models: 200 k).
4. **`truncate_to_fit`** — automatically prune the oldest turns to keep a chat within budget while preserving the system prompt and the latest user message.
5. **Extended thinking** — give Claude a reasoning budget for problems that benefit from careful deliberation; bill as output tokens.
6. **`response.usage`** — four fields (`input_tokens`, `output_tokens`, `cache_read_input_tokens`, `cache_creation_input_tokens`) for precise accounting after every call.
7. **`CostEstimator`** — translate token counts into dollar amounts and accumulate a session total.

Combining these building blocks gives you full visibility and control over context and cost in any Claude-powered application.